Data Ingestion and Filtering

# Phase 1: Generalizable Single-Cell Streaming & Target Data Ingestion

## Executive Overview
This notebook establishes the upstream data conveyor link for this digital biology pipeline. It streams high-throughput functional genomics datasets from the **Xaira Therapeutics X-Atlas/Orion platform** using memory-optimized lazy streaming evaluation. 

The data parsing design is fully decoupled and generalizable; by altering the global boundary properties, researchers can adapt this pipeline to target any arbitrary HGNC gene symbol configuration.

### Key Objectives
1. **Lazy Loading Streams:** Connect directly to Hugging Face dataset registries using data streaming layers to mitigate localized memory constraints.
2. **Dynamic Parametrization:** Avoid hardcoded string masks by abstracting target genetic components to global variables.
3. **Serialization Handover:** Extract, validate, and write targeting array buffers onto disk via object stream pickles.

In [ ]:
import os
import pickle
#from datasets import load_dataset

# =====================================================================
# GLOBAL SEED CONFIGURATION BOUNDARY (Generalizable Target)
# =====================================================================
# To target a different genetic axis, simply modify these two properties.
# Standard defaults configured for human Gout transporter targets:
TARGET_GENE_SYMBOL = "ABCG2" 
TARGET_ENSEMBL_ID  = "ENSG00000118777"
CELL_LINE_SPLIT    = "HEK293T"

# Pipeline scaling bounds
MAX_SUBSET_CELLS   = 35 
OUTPUT_DIRECTORY   = "data"
OUTPUT_FILENAME    = "gout_cells_subset.pkl"

print(f"Pipeline parameters initialized for target: {TARGET_GENE_SYMBOL} ({TARGET_ENSEMBL_ID})")

📡 Pipeline parameters initialized for target: ABCG2 (ENSG00000118777)


In [2]:
print(os.getcwd())

c:\Users\User\OneDrive\Desktop\NUS\MSc AIS\gout-transcriptome-causal\gout-transcriptome-causal\notebooks


## Establishing Lazy Registry Connections
Rather than downloading the massive gigabyte-scale multi-cell array collections directly onto our ephemeral cloud node, we configure an active internet streaming cursor using `streaming=True`. This processes elements one memory slice at a time.

In [ ]:
# Initialize a lazy streaming iterator loop pointing to the Xaira registry split
print(f"Connecting to Xaira X-Atlas-Orion [{CELL_LINE_SPLIT}] streaming split...")
dataset_stream = load_dataset(
    "Xaira-Therapeutics/X-Atlas-Orion", 
    streaming=True, 
    split=CELL_LINE_SPLIT
)

# Pull a lone row matrix out of the stream buffer to introspect schema structures
stream_iterator = iter(dataset_stream)
introspected_row = next(stream_iterator)

print("\nUpper Registry Schema Introspection (Available Keys):")
for key in sorted(introspected_row.keys()):
    print(f"  └── key: {key}")

## Running The Automated Target Extraction
With the schema structure verified, the cursor walks through individual single-cell screening layers to find expressions targeted by our configured guide tracking configurations.

In [ ]:
filtered_cells = []
print(f"Searching active stream arrays for perturbations targeting: {TARGET_GENE_SYMBOL}...")

for idx, cell in enumerate(dataset_stream):
    # Dynamic dictionary comparison using our global boundary key configuration
    # This ensures no single string literal is hardcoded downstream
    if cell.get('gene_target') == TARGET_GENE_SYMBOL:
        filtered_cells.append(cell)
        print(f"  [+] Match found! Total subset buffer pool: {len(filtered_cells)} cells.")

    # Terminate the lazy stream ingestion once the target threshold envelope is met
    if len(filtered_cells) >= MAX_SUBSET_CELLS:
        break

print(f"\nExtraction Threshold Achieved. Successfully compiled {len(filtered_cells)} cellular profiles.")

## Local Volume Disk Handover
We serialize the in-memory array data onto disk as a binary stream checkpoint. This acts as the official data handoff file for **Phase 2: Preprocessing**.

In [ ]:
# Ensure the local data folder environment exists 
os.makedirs(OUTPUT_DIRECTORY, exist_ok=True)
handover_path = os.path.join(OUTPUT_DIRECTORY, OUTPUT_FILENAME)

print(f"📦 Serializing {len(filtered_cells)} cellular objects to: {handover_path}")
with open(handover_path, 'wb') as file_handler:
    pickle.dump(filtered_cells, file_handler)

print("🚀 Volume handover completely established. Stream ingestion gracefully unmounted.")